In [2]:
from ultralytics import YOLO
import torch
from PIL import Image

# --- Начало обходного пути для PyTorch ---
# Сохраняем оригинальную функцию torch.load
_original_torch_load = torch.load

# Определяем новую функцию, которая вызывает оригинальную с weights_only=False
def _custom_torch_load(*args, **kwargs):
    # Устанавливаем weights_only в False, если ключ отсутствует или равен None
    kwargs.setdefault('weights_only', False)
    if kwargs['weights_only'] is not False:
        kwargs['weights_only'] = False
        
    return _original_torch_load(*args, **kwargs)

# Временно заменяем torch.load нашей версией
torch.load = _custom_torch_load
# --- Конец обходного пути ---

# Загрузка модели. Проверьте имя файла, возможно, должно быть "yolov8n-seg.pt"
model = YOLO("yolo11n-seg.pt")  # load an official segmentation model

# --- Восстановление оригинальной функции torch.load ---
torch.load = _original_torch_load
# --- Конец восстановления ---

# Predict with the model
results = model("../data/fridge.jpg")  # predict on an image
im_array = results[0].plot()
im = Image.fromarray(im_array[..., ::-1])  # BGR to RGB
im.show()

# Access the results
for result in results:
    if result.masks:
        xy = result.masks.xy  # mask in polygon format
        xyn = result.masks.xyn  # normalized
        masks = result.masks.data  # mask in matrix format (num_objects x H x W)


image 1/1 /Users/infibiss/Desktop/Fridge-Recipe-Detector/models/../data/fridge.jpg: 608x640 1 bottle, 1 bowl, 2 apples, 2 oranges, 1 vase, 86.2ms
Speed: 6.1ms preprocess, 86.2ms inference, 4.8ms postprocess per image at shape (1, 3, 608, 640)
